# Google Drive OAuth2 Token Generator
This notebook helper will guide you through the process of generating a Google Drive OAuth2 token (`GD_USER_TOKEN_JSON`) which can be used to authenticate Google Drive operations (like uploading datasets or checkpoints) from Kaggle or other headless environments using your personal account's storage quota.

## Prerequisites:
1. Go to the [Google Cloud Console](https://console.cloud.google.com/).
2. Select or create a Google Cloud Project.
3. Search for **Google Drive API** in the API Library and enable it for your project.
4. Go to **APIs & Services > OAuth consent screen**:
   - Choose User Type **External**.
   - Fill in the required fields (App name, support email, developer contact email).
   - Under **Scopes**, you can leave it blank for now, or add `.../auth/drive`.
   - Under **Test users**, add the email address of the Google account whose Drive you want to use. **(CRITICAL: If you don't do this, Google will block you with access_denied!)**
   - Save and finish.
5. Go to **APIs & Services > Credentials**:
   - Click **Create Credentials** and choose **OAuth client ID**.
   - Select application type **Desktop app** (or Desktop application).
   - Name it (e.g. "Kaggle Drive Uploader") and click **Create**.
   - Click the download icon (Download JSON) next to your newly created Client ID.
6. Open the downloaded JSON file, copy its entire contents, and paste it into the cell below.

In [ ]:
# ── Paste your OAuth Client Secrets JSON here ──────────────────────────────────
# Paste the content of the downloaded OAuth client secret JSON file below:
client_secrets_json = """
{
  "installed": {
    "client_id": "YOUR_CLIENT_ID.apps.googleusercontent.com",
    "project_id": "YOUR_PROJECT_ID",
    "auth_uri": "https://accounts.google.com/o/oauth2/auth",
    "token_uri": "https://oauth2.googleapis.com/token",
    "auth_provider_x509_cert_url": "https://www.googleapis.com/oauth2/v1/certs",
    "client_secret": "YOUR_CLIENT_SECRET",
    "redirect_uris": [
      "http://localhost"
    ]
  }
}
"""

## Generate OAuth Token
Run the cell below. It will:
1. Initialize the OAuth flow using the client configuration you pasted.
2. Generate an authorization URL.
3. Prompt you to open the URL in your browser, log in with your Google account, and grant access to Google Drive.
4. Google will show an error screen saying the app is not verified (since you created it yourself). Click **Advanced** and then **Go to ... (unsafe)** to proceed.
5. After granting permissions, Google will redirect you to a localhost address (e.g. `http://localhost/?code=4/0A...`).
6. Copy the **entire redirect URL** from your browser's address bar (or just the value of the `code=` parameter) and paste it into the prompt box below.

In [ ]:
import json
import urllib.parse
from google_auth_oauthlib.flow import InstalledAppFlow

# Parse client configuration
try:
    client_config = json.loads(client_secrets_json.strip())
except Exception as e:
    raise ValueError(f"Failed to parse client secrets JSON. Please make sure you pasted the exact content from the file. Error: {e}")

# Configure scopes
scopes = ["https://www.googleapis.com/auth/drive"]

# Create flow helper
flow = InstalledAppFlow.from_client_config(client_config, scopes=scopes)

# Step 1: Generate Authorization URL
authorization_url, state = flow.authorization_url(
    access_type="offline",
    include_granted_scopes="true",
    prompt="consent"
)

print("=" * 80)
print("1. Click the link below to authorize access:")
print(authorization_url)
print("=" * 80)

# Step 2: Prompt for authorization code or redirect URL
print("\nAfter authorizing, your browser will redirect to a localhost URL (which may display a 'Site can't be reached' page).")
print("Copy the ENTIRE URL from the browser's address bar (e.g. http://localhost/?code=4/0A...&scope=...)")
print("and paste it below.")

redirect_response = input("\nPaste the redirect URL or authorization code here: ").strip()

# If they just pasted the code, reconstruct the redirect response
if redirect_response.startswith("4/"):
    code = redirect_response
else:
    parsed = urllib.parse.urlparse(redirect_response)
    query_params = urllib.parse.parse_qs(parsed.query)
    code = query_params.get("code", [redirect_response])[0]

# Step 3: Exchange authorization code for token
print("\nExchanging code for credentials...")
flow.fetch_token(code=code)
credentials = flow.credentials

# Step 4: Output token JSON
print("\n" + "=" * 80)
print("  GENERATION SUCCESSFUL!")
print("=" * 80)
print("\nCopy the ENTIRE JSON block below (including the outer braces):\n")
print(credentials.to_json())
print("\n" + "=" * 80)
print("Add this JSON block as a Secret named 'GD_USER_TOKEN_JSON' in your Kaggle Notebook's Add-ons > Secrets.")
print("=" * 80)
